### This notebook is used to calculate the final partial dependence plot values for the most important features: the values are saved on box-cox scale and also retransformed to original scale

input files: 

    * optimized_model_parms_for_importance_calculation.csv (containing optimized model parameters)
    * median_cnp_export_abs_conc_and_rfr_and_basin_feat.csv (containing dataset features and targets)

output files:

    * partial dependence values for the most important features

settings: 

    * target var must be set: n_imbal, c_imbal, p_imbal, median_DIN, median_SRP or median_DOC_TOC_bioav
    * model type must be defined: either gbr (GradientBoostingRegressor) or lightgbm (LightGBMRegressor)

# Load required modules: 

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.transforms
from sklearn import metrics
from numpy import mean
from numpy import std
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RepeatedKFold
import time
import os
import ast
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
import os
from sklearn.preprocessing import PowerTransformer
import pdb
import time

In [2]:
# Loading Data
data = pd.read_csv('../../../output_data/input_ML_learning/median_cnp_export_abs_conc_and_rfr_and_basin_feat.csv').drop(columns='Unnamed: 0')
optimized_parameter_sets = pd.read_csv('../../../output_data/ML_analysis/hyperparameter_settings/optimized_model_parms_for_importance_calculation.csv', sep = ';', dtype={'parameters': str})
# Convert the 'parameters' column to dictionaries
optimized_parameter_sets['parameters'] = optimized_parameter_sets['parameters'].apply(ast.literal_eval)
# Define the directory path


In [3]:
def create_input_ML(data, target_var):

    # create array with exported nutrient values, which are our target values, we want to predict 
    target = np.array(data[target_var])
    # remove the labels from the features df:
    features = data.drop(['n_imbal', 'p_imbal', 'c_imbal', 'median_DOC_TOC_bioav', 'median_DIN', 'median_SRP'], axis = 1)
    #features = nutrient_export_data.filter(items=selected_features)

    #select columns by index: 
    features= features.iloc[:,0:43]
    features.columns

    # save feature list for later use:
    feature_list = list(features.columns)

    # convert to numpy array:
    #features = np.array(features)

    return target, features, feature_list

In [4]:
#select model type: 'gbr' or 'lightgbm'
model_type = 'gbr'
#set random_state:
run_number = [1,2,3,4,5]
random_state_split = [2, 19, 44, 33, 21]
random_state_regressor = [4, 9, 19, 31, 41]
random_state_feature_importances = [1, 17, 45, 22, 12]

### Now fit for all five splits the models:

In [5]:
# select target_var: n_imbal, c_imbal, p_imbal:
#target_var = 'p_imbal'

# create empty data lists to save data:
# test_features_runs
t0 = time.time()
df_outputs = []
df_metrics = []
transf_bc = {}
lambda_values = {}
pt_dict = {}
train_targets_splits = {}
train_features_splits = {}
test_features_splits = {}

models_splits = {}
boxcox = True
stratified_sampling = False
for target_var in ['p_imbal','c_imbal','n_imbal']:  
    train_targets_splits[target_var] = [] 
    train_features_splits[target_var] = []
    lambda_values[target_var] = {}
    pt_dict[target_var] = {}
    target, features, feature_list = create_input_ML(
        data = data, target_var = target_var)
    # extract best parameters: squeeze() funtion is required, to convert Series object to one value (result of loc operation is always a pandas series, even if output is only one value):
    best_parms =  optimized_parameter_sets.loc[
        (optimized_parameter_sets['target'] == target_var) & (optimized_parameter_sets['model'] == model_type),
        'parameters'].squeeze()
    transf_bc[target_var] = []
    test_features_splits[target_var] = []
    r2_train_splits = []
    r2_test_splits = []
    models_splits[target_var] = []
    pred_test_bc_df = pd.DataFrame()
    pred_test_original_scale_df = pd.DataFrame()
    obs_test_bc_df = pd.DataFrame()
    obs_test_original_scale_df = pd.DataFrame()
    
    # loop to split the data into training and test data and to fit the model: using diffent random states:
    for i, (split, regressor, feature) in enumerate(zip(random_state_split, random_state_regressor, random_state_feature_importances)):       
        if stratified_sampling:
            num_quantiles = 20  # Adjust this to the number of quantiles you want
            target_binned = pd.qcut(target, q=num_quantiles, labels=False)
            train_features, test_features, train_targets, test_targets = train_test_split(
                features, target, test_size=0.20, random_state=split, stratify = target_binned)
        else:
            train_features, test_features, train_targets, test_targets = train_test_split(features, target, test_size=0.20, random_state=split)
        
        train_IDs = train_features['HYBAS_ID'].copy()
        test_IDs = test_features['HYBAS_ID'].copy()
        train_features =train_features.drop('HYBAS_ID',axis=1)
        test_features = test_features.drop('HYBAS_ID',axis=1)
            
        if len(train_targets.shape) == 1:
            train_targets_flat = np.reshape(train_targets, (-1, 1))
        if len(test_targets.shape) == 1:
            test_targets_flat = np.reshape(test_targets, (-1, 1))
        if boxcox:
            pt = PowerTransformer(method='box-cox')
            # Reshape train_targets to 2D if it's 1D
            # Fit the transformer to the training data
            # This learns the best lambda value for each feature
            pt.fit(train_targets_flat)
            lambda_values[target_var][i] = pt.lambdas_
            pt_dict[target_var][i] = pt
            # Transform the training data using the learned lambda values
            train_targets_transformed_bc = pt.transform(train_targets_flat)
            # convert back to 1D format
            train_targets_transformed_bc = train_targets_transformed_bc.ravel()

            # Transform the test data using the same lambda values
            # Reshape test_targets to 2D if it's 1D
            test_targets_transformed_bc = pt.transform(test_targets_flat)
            test_targets_transformed_bc = test_targets_transformed_bc.ravel()
            train_targets_to_model = train_targets_transformed_bc
        else:
            train_targets_to_model = train_targets_flat
        
        # save test_features to list:
        test_features_splits[target_var].append(test_features)
        train_targets_splits[target_var].append(train_targets_transformed_bc)
        train_features_splits[target_var].append(train_features)

        # Fit the model:   
        model = GradientBoostingRegressor(**best_parms,  random_state = regressor, validation_fraction = 0.1, n_iter_no_change = 10)
        model.fit(train_features, train_targets_to_model)
        # save the model to a list:
        models_splits[target_var].append(model)

        # make predictions on train- and test-features:
        pred_train = model.predict(train_features)
        pred_test = model.predict(test_features)
        pred_test_bc = pred_test.reshape(-1, 1)
        
        obs_test_original_scale = test_targets.reshape(-1, 1)
        df = pd.DataFrame()
        metrics = pd.DataFrame()
        if boxcox:
            # transform back to original scale:
            pred_train_original_scale = pt.inverse_transform(pred_train.reshape(-1, 1))
            pred_test_original_scale = pt.inverse_transform(pred_test.reshape(-1, 1))
            obs_test_bc = test_targets_transformed_bc.reshape(-1, 1)
            df['obs_test_norm'] = obs_test_bc.flatten().copy()
            df['preds_norm'] = pred_test_bc.flatten().copy()
            r2_train = r2_score(train_targets_transformed_bc, pred_train)
            r2_test = r2_score(test_targets_transformed_bc, pred_test)
            print(f"R2 Test boxcox {target_var,i}: {r2_test}")
            metrics.loc[0,'R2_test_boxcox'] = r2_test
        else:
            pred_test_original_scale = pred_test_bc

        df['obs_test'] = obs_test_original_scale.flatten().copy() 
        df['preds_test'] = pred_test_original_scale.flatten().copy()
        df['nrand'] = i
        df['comp'] = target_var
        df['station'] = test_IDs.values
        df_outputs.append(df.copy())

        r2_train_orig = r2_score(train_targets, pred_train_original_scale)
        r2_test_orig = r2_score(test_targets, pred_test_original_scale)
        #print(f"R2 Train boxcox {target_var,i}: {r2_train}")
        #print(f"R2 Train original scale {target_var,i}: {r2_train_orig}")
        print(f"R2 Test original scale {target_var,i}: {r2_test_orig}")
        metrics.loc[0,'R2_test'] = r2_test_orig
        metrics.loc[0,'nrand'] = i
        metrics.loc[0,'comp'] = target_var
        df_metrics.append(metrics.copy())
        #r2_train_splits.append(r2_train)
        #r2_test_splits.append(r2_test)
        
    print('Elapsed time',time.time()-t0)
df_outputs = pd.concat(df_outputs,axis=0)

R2 Test boxcox ('p_imbal', 0): 0.384237053039312
R2 Test original scale ('p_imbal', 0): 0.3333361091884026
R2 Test boxcox ('p_imbal', 1): 0.3771722450259485
R2 Test original scale ('p_imbal', 1): 0.33054802072330913
R2 Test boxcox ('p_imbal', 2): 0.38586426485382574
R2 Test original scale ('p_imbal', 2): 0.3289651247125325
R2 Test boxcox ('p_imbal', 3): 0.33551627539187723
R2 Test original scale ('p_imbal', 3): 0.24340957989220824
R2 Test boxcox ('p_imbal', 4): 0.3699119975296674
R2 Test original scale ('p_imbal', 4): 0.3265525715918717
Elapsed time 34.789902687072754
R2 Test boxcox ('c_imbal', 0): 0.6540695670853867
R2 Test original scale ('c_imbal', 0): 0.4494863396029237
R2 Test boxcox ('c_imbal', 1): 0.5724326466072911
R2 Test original scale ('c_imbal', 1): 0.45053444381988095
R2 Test boxcox ('c_imbal', 2): 0.6018280249981082
R2 Test original scale ('c_imbal', 2): 0.38333802213534773
R2 Test boxcox ('c_imbal', 3): 0.5628681068630688
R2 Test original scale ('c_imbal', 3): 0.43120888

In [6]:
### Create a dictionary containing target variables units: slp, dor, temperature and hdi were converted:
feature_var_units = {
    'UP_AREA':'[km$^{2}$]' ,
    'twi90':'[-]', 
    'slp_dg_uav':'[°]',
    'for_pc_use':'[%]',
    'crp_pc_use':'[%]',
    'pst_pc_use':'[%]',
    'ppd_pk_uav':'[people km$^{-2}$]',
    'run_mm_syr':'[mm year$^{-1}$]',
    'inu_pc_umn':'[%]',
    'dor_pc_pva':'[%]',
    'ria_ha_usu_perkm2':'[hectares km$^{-2}$]', 
    'riv_tc_usu_perkm2':'[hectares km$^{-2}$]',
    'ele_mt_uav':'[m a.s.l]',
    'sgr_dk_sav':'[dm km$^{-1}$]',
    'tmp_dc_uyr':'[°C]', 
    'pre_mm_uyr':'[mm year$^{-1}$]',
    'pet_mm_uyr':'[mm year$^{-1}$]',
    'aet_mm_uyr':'[mm year$^{-1}$]',
    'snw_pc_uyr':'[%]',
    'wet_pc_ug2':'[%]',
    'ire_pc_use':'[%]',
    'gla_pc_use':'[%]',
    'prm_pc_use':'[%]',
    'pac_pc_use':'[%]',
    'cly_pc_uav':'[%]',
    'slt_pc_uav':'[%]',
    'snd_pc_uav':'[%]',
    'soc_th_uav':'[tonnes hectare$^{-1}$]',
    'swc_pc_uyr':'[%]',
    'kar_pc_use':'[%]',
    'ero_kh_uav':'[kg hectare$^{-1}$ year$^{-1}$]',
    'gdp_ud_usu':'[US dollers]',
    'hdi_ix_sav':'[-]'
}
#perm_imp_features = perm_imp_features.tolist()

### 1.  Calculate partial dependence plots for the most important features: each for all five different train_test splits:

In [8]:
#RUNNING PARTIAL DEPENDENCE PLOTS PARALLEL

from sklearn.inspection import partial_dependence
from joblib import Parallel, delayed
import matplotlib.pyplot as plt
import time

# Initialize a dictionary to store all results
perm_imp_features = ['swc_pc_uyr', 'pet_mm_uyr', 'hdi_ix_sav', 'wet_pc_ug2','ppd_pk_uav']
tic = time.time()
target_vars = ['p_imbal', 'c_imbal', 'n_imbal']
all_pdp_results_splits = {target_var: {} for target_var in target_vars}

# Define a function to calculate partial dependence for each feature
def compute_pdp(model, train_features, feature, grid_resolution=100):
    pdp_results = partial_dependence(
        model, train_features, [feature], method='brute', kind='both',
        percentiles=(0.05, 0.95), grid_resolution=grid_resolution
    )
    return pdp_results

# Run parallel computation of partial dependence
for target_var in target_vars: 
    all_pdp_results_splits[target_var] = {}
    for n, model in enumerate(models_splits[target_var]):
        # Use joblib to parallelize the loop over features
        all_pdp_results_splits[target_var][n] = {}
        pdp_results = Parallel(n_jobs=-1)(
            delayed(compute_pdp)(
                model, train_features_splits[target_var][n], feature
            ) for feature in perm_imp_features
        )

        # Store results in the dictionary
        for i, feature in enumerate(perm_imp_features):
            all_pdp_results_splits[target_var][n][feature] = pdp_results[i]
            print(f"Completed PDP for {target_var}, model {n}, feature {feature}, time elapsed: {time.time() - tic:.2f} seconds")

# Optional: Plotting logic here

Completed PDP for p_imbal, model 0, feature swc_pc_uyr, time elapsed: 21.74 seconds
Completed PDP for p_imbal, model 0, feature pet_mm_uyr, time elapsed: 21.74 seconds
Completed PDP for p_imbal, model 0, feature hdi_ix_sav, time elapsed: 21.74 seconds
Completed PDP for p_imbal, model 0, feature wet_pc_ug2, time elapsed: 21.74 seconds
Completed PDP for p_imbal, model 0, feature ppd_pk_uav, time elapsed: 21.74 seconds
Completed PDP for p_imbal, model 1, feature swc_pc_uyr, time elapsed: 48.11 seconds
Completed PDP for p_imbal, model 1, feature pet_mm_uyr, time elapsed: 48.11 seconds
Completed PDP for p_imbal, model 1, feature hdi_ix_sav, time elapsed: 48.11 seconds
Completed PDP for p_imbal, model 1, feature wet_pc_ug2, time elapsed: 48.11 seconds
Completed PDP for p_imbal, model 1, feature ppd_pk_uav, time elapsed: 48.11 seconds
Completed PDP for p_imbal, model 2, feature swc_pc_uyr, time elapsed: 73.84 seconds
Completed PDP for p_imbal, model 2, feature pet_mm_uyr, time elapsed: 73.84 

### Retransform the partial dependence results to the original scale:

* final_df comprises average partial dependence values retransformed to original scale: for all target_vars (c_imbal, n_imbal and p_imbal), and all five selected features
 
* individuals_df is a dictionary comrising all individual partial dependencies for all catchments individual

In [9]:
from scipy.special import inv_boxcox
final_df = []
individuals_df = {target_var:{} for target_var in target_vars}
for target_var in target_vars:  
    individuals_df[target_var] = {}
    for n, _ in enumerate(models_splits[target_var]):
        individuals_df[target_var][n] = {}
        for i, feature in enumerate(perm_imp_features):
            
            feature_unit = feature_var_units[feature]
            # Add baseline in Box-Cox space to PDP values, then inverse transform
            pdp_values_transformed = all_pdp_results_splits[target_var][n][feature]['average'][0] 
            #pdp_values_original_space = [pt_dict[target_var][n].inverse_transform(values.reshape(-1,1)) for values in pdp_values_transformed]
            pdp_values_original_space = pt_dict[target_var][n].inverse_transform(pdp_values_transformed.reshape(-1,1)) 

            df_ = pd.DataFrame()
            df_ind = pd.DataFrame()
            df_['feature_val'] = all_pdp_results_splits[target_var][n][feature]['grid_values'][0].ravel()
            df_['partial_dep'] = np.array(pdp_values_original_space).ravel()
            df_['var'] = target_var
            df_['nrand'] = n
            df_['feature_name'] = feature
            individuals_df[target_var][n][feature] = pd.DataFrame(all_pdp_results_splits[target_var][n][feature]['individual'][0])
            final_df.append(df_.copy())
        #print(pt_dict[target_var][n].lambdas_)
final_df = pd.concat(final_df,axis=0)
final_df = final_df.set_index(['var','feature_name','nrand']).sort_index()


In [10]:
individuals_df[target_var][n].keys()

dict_keys(['swc_pc_uyr', 'pet_mm_uyr', 'hdi_ix_sav', 'wet_pc_ug2', 'ppd_pk_uav'])

In [11]:
final_df.loc['c_imbal','ppd_pk_uav',0]	

feature_val  partial_dep
var     feature_name nrand                          
c_imbal ppd_pk_uav   0         2.847780     0.049951
                     0         7.358425     0.048367
                     0        11.869069     0.044591
                     0        16.379714     0.043604
                     0        20.890358     0.041494
...                                 ...          ...
                     0       431.359012     0.026770
                     0       435.869656     0.026730
                     0       440.380301     0.026688
                     0       444.890945     0.026657
                     0       449.401590     0.026659

[100 rows x 2 columns]

In [12]:
individuals_df[target_var][n][feature]

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,0.084533,0.137568,0.241581,0.290718,0.393644,0.413919,0.427939,0.441258,0.454611,0.473876,...,0.438545,0.438545,0.438545,0.437284,0.437284,0.438688,0.438688,0.437006,0.436643,0.436658
1,-0.788697,-0.820354,-0.815785,-0.780846,-0.799694,-0.809340,-0.807741,-0.807419,-0.811586,-0.804139,...,-0.772294,-0.773033,-0.778615,-0.781570,-0.782454,-0.782409,-0.781614,-0.779483,-0.778401,-0.776847
2,-0.877426,-0.880926,-0.858285,-0.847800,-0.853127,-0.870200,-0.881877,-0.896783,-0.899441,-0.909641,...,-0.923404,-0.923404,-0.925348,-0.925348,-0.925323,-0.922420,-0.922420,-0.922172,-0.922172,-0.922172
3,-1.410824,-1.443980,-1.492476,-1.524631,-1.580687,-1.614260,-1.635598,-1.636265,-1.626114,-1.614194,...,-1.533861,-1.531288,-1.531488,-1.531488,-1.531488,-1.524708,-1.524708,-1.522975,-1.522975,-1.522975
4,-0.489211,-0.460594,-0.429853,-0.409765,-0.371405,-0.364142,-0.360111,-0.360550,-0.349782,-0.344893,...,-0.290979,-0.283056,-0.283056,-0.282259,-0.281142,-0.263873,-0.263873,-0.263873,-0.266965,-0.265534
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2791,-0.667933,-0.649516,-0.574818,-0.543089,-0.497818,-0.483743,-0.475169,-0.463816,-0.461964,-0.464606,...,-0.587529,-0.587529,-0.587529,-0.587319,-0.587319,-0.587474,-0.587474,-0.587961,-0.582640,-0.582185
2792,0.358039,0.408788,0.500037,0.534179,0.561685,0.554719,0.560866,0.558782,0.538189,0.546451,...,0.436845,0.436845,0.436845,0.437576,0.437576,0.437698,0.437698,0.436090,0.435728,0.435152
2793,-0.412038,-0.407839,-0.403619,-0.412898,-0.407730,-0.426499,-0.434860,-0.443512,-0.467423,-0.470829,...,-0.448433,-0.448433,-0.448433,-0.448433,-0.453039,-0.444655,-0.444655,-0.444655,-0.444655,-0.442737
2794,-0.278093,-0.248513,-0.218835,-0.234862,-0.239871,-0.252525,-0.252440,-0.253024,-0.261869,-0.261128,...,-0.279160,-0.279160,-0.279160,-0.276179,-0.274220,-0.264194,-0.264194,-0.262095,-0.262095,-0.262095


In [13]:
aux_c = final_df.loc['c_imbal'].rename(columns={'feature_val':'feature','partial_dep':'c_imbal'})
aux_p = final_df.loc['p_imbal'].rename(columns={'feature_val':'feat_c','partial_dep':'p_imbal'})
aux_n = final_df.loc['n_imbal'].rename(columns={'feature_val':'feat_c','partial_dep':'n_imbal'})
df_pdp = pd.concat([aux_c,aux_p,aux_n],axis=1)[['feature','c_imbal','p_imbal','n_imbal']]
df_pdp.index.unique()

MultiIndex([('hdi_ix_sav', 0),
            ('hdi_ix_sav', 1),
            ('hdi_ix_sav', 2),
            ('hdi_ix_sav', 3),
            ('hdi_ix_sav', 4),
            ('pet_mm_uyr', 0),
            ('pet_mm_uyr', 1),
            ('pet_mm_uyr', 2),
            ('pet_mm_uyr', 3),
            ('pet_mm_uyr', 4),
            ('ppd_pk_uav', 0),
            ('ppd_pk_uav', 1),
            ('ppd_pk_uav', 2),
            ('ppd_pk_uav', 3),
            ('ppd_pk_uav', 4),
            ('swc_pc_uyr', 0),
            ('swc_pc_uyr', 1),
            ('swc_pc_uyr', 2),
            ('swc_pc_uyr', 3),
            ('swc_pc_uyr', 4),
            ('wet_pc_ug2', 0),
            ('wet_pc_ug2', 1),
            ('wet_pc_ug2', 2),
            ('wet_pc_ug2', 3),
            ('wet_pc_ug2', 4)],
           names=['feature_name', 'nrand'])

### Retransform individual pdp values to original scale:

In [14]:
individuals_df_orig_scale = {target_var:{} for target_var in target_vars}
for target_var in target_vars:  
    individuals_df_orig_scale[target_var] = {}
    for n, _ in enumerate(models_splits[target_var]):
        individuals_df_orig_scale[target_var][n] = {}
        for i, feature in enumerate(perm_imp_features):

            df_auxa = individuals_df[target_var][n][feature]
            df_retransformed = pd.DataFrame()
            for column in df_auxa.columns:
                col_orig_scle = pd.DataFrame(pt_dict[target_var][n].inverse_transform(df_auxa[column].values.reshape(-1,1)))
                df_retransformed = pd.concat([df_retransformed, col_orig_scle], axis=1)

            #set column names    
            df_retransformed.columns = df_auxa.columns
            # append retransformed data to dictionary
            individuals_df_orig_scale[target_var][n][feature] = df_retransformed



### Select for each model run, each target variable and each feature the min and maxium value:

In [15]:
# create an empty dataframe for the min and max partial dpendence values:
df_pdp_min_max_values_all = pd.DataFrame()

# loop over all target variables, models and features:
for target_var in target_vars:
    for n, _ in enumerate(models_splits[target_var]):
        for i, feature in enumerate(perm_imp_features):
            # extract  min and max values of the individual pdp values nd their indices:
            df_temp = pd.DataFrame()
            min_pdp_values = individuals_df_orig_scale[target_var][n][feature].iloc[:,0]
            max_pdp_values = individuals_df_orig_scale[target_var][n][feature].iloc[:,-1]
            combined_pdp_values = pd.concat([min_pdp_values, max_pdp_values], ignore_index=True)
            combined_catch_index = min_pdp_values.index.tolist() + max_pdp_values.index.tolist() 
            df_temp['catch_ind'] = combined_catch_index  

            # get the corresonding min and max values of the feature:
            feature_min_val = final_df.loc[target_var].loc[feature].loc[n]['feature_val'].min()
            feature_max_val = final_df.loc[target_var].loc[feature].loc[n]['feature_val'].max()
            combined_feature_values = [feature_min_val] * len(min_pdp_values) + [feature_max_val] * len(max_pdp_values) 
            
            # df_temp:

            df_temp = pd.DataFrame({
                'catch_ind' : combined_catch_index,
                'pdp_val' : combined_pdp_values,
                'feature_val' : combined_feature_values,
                'var' : target_var,
                'feature' : feature,
                'nrand' : n
            })
            #df_temp['catch_ind'] = combined_catch_index
            #df_temp['pdp_val'] = combined_pdp_values
            #df_temp['feature_val'] = combined_feature_values
            #df_temp['var'] = pd.Series([target_var]*len(df_temp))   
            #df_temp['feature'] = pd.Series([feature]*len(df_temp))
            #df_temp['nrand'] = pd.Series([n]*len(df_temp))  
            
            df_pdp_min_max_values_all = pd.concat([df_pdp_min_max_values_all, df_temp], ignore_index=True)
            


    

### Pivot table: 
* group df_pdp_min_max_values_all by catch_ind, feature_val, feature, nrand and var (target), and subsequently make for each target variable one column with the corresonding pdp value:

In [16]:
df_pivoted = (
    df_pdp_min_max_values_all.groupby(['catch_ind', 'feature_val', 'feature', 'nrand', 'var'])['pdp_val']
      .first()  # Use 'first' if you only want one pdp_val per var; adjust as needed.
      .unstack('var')  # Pivot to make each 'var' a column
      .reset_index()  # Reset index to flatten the DataFrame
)

df_pivoted.loc[df_pivoted['catch_ind'] == 0].loc[df_pivoted['feature'] == 'ppd_pk_uav']

var,catch_ind,feature_val,feature,nrand,c_imbal,n_imbal,p_imbal
5,0,2.84778,ppd_pk_uav,0,0.037576,0.557476,0.400705
6,0,2.84778,ppd_pk_uav,3,0.119868,0.555833,0.289872
7,0,2.94605,ppd_pk_uav,1,0.050556,0.760226,0.135815
8,0,2.98394,ppd_pk_uav,4,0.066147,0.779166,0.116404
9,0,3.16431,ppd_pk_uav,2,0.091019,0.719885,0.151144
25,0,449.40159,ppd_pk_uav,0,0.007180,0.544854,0.427470
26,0,473.90612,ppd_pk_uav,1,0.017413,0.821571,0.143332
27,0,477.91301,ppd_pk_uav,4,0.032285,0.832874,0.116624
28,0,480.17195,ppd_pk_uav,2,0.047823,0.766323,0.144188
29,0,482.71848,ppd_pk_uav,3,0.085309,0.574641,0.320759


### Save dataframe with min and max pdp values for all 5 perm_imp_features and all models and target variables:

In [19]:
# define the output folder path:

output_folder = '../../../output_data/ML_analysis/partial_dependence_plots_paper'

# Check if the folder exists, if not, create it:
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

In [20]:
df_pivoted.to_csv('../../../output_data/ML_analysis/partial_dependence_plots_paper/individual_pdp_values_original_scale_min_max.csv', index = False)

### save pdp data in box cox and  original space as pkl files:

In [21]:
import pickle

### at first Box-Cox scaled partial dependence plot values for all models and target variables:
# Save as pickle
with open(f"../../../output_data/ML_analysis/partial_dependence_plots_paper/pdp_results_box_cox_scale.pkl", "wb") as f:
    pickle.dump(all_pdp_results_splits, f)


### partial dependence plot average values, retransformed to original scale for all models and target variables:	 
# Save as pickle
with open(f"../../../output_data/ML_analysis/partial_dependence_plots_paper/pdp_results_original_scale_average.pkl", "wb") as f:
    pickle.dump(final_df, f)


### individual partial dependence plots on Box-Cox scale for all models and target variables:	
with open(f"../../../output_data/ML_analysis/partial_dependence_plots_paper/individuals_df_pdp_results_dic_box_cox.pkl", "wb") as f:
    pickle.dump(individuals_df, f)

### individual partial dependence plots retransformed to original scale for all models and target variables:	
with open(f"../../../output_data/ML_analysis/partial_dependence_plots_paper/individuals_df_pdp_results_dic_original_scale.pkl", "wb") as f:
    pickle.dump(individuals_df_orig_scale, f)

In [22]:
final_df.to_csv(f"../../../output_data/ML_analysis/partial_dependence_plots_paper/pdp_results_original_scale_average.csv")